In [26]:
import os
import sys
import sys

sys.path.append(os.path.abspath('..'))

WORKSPACE_DIR = "../lightrag_graphs"
os.makedirs(WORKSPACE_DIR, exist_ok=True)
ARCHIVO_ACTUAL = "../data/raw/book/christmas_carol_all.txt"

In [25]:
import os
import asyncio
import numpy as np
from openai import AsyncOpenAI
from lightrag import LightRAG
from lightrag.utils import wrap_embedding_func_with_attrs
from lightrag.llm.openai import openai_complete_if_cache
import nest_asyncio
nest_asyncio.apply()

In [27]:
print("⚙️ Configurando LightRAG con motor 100% Gemini...")

# CAMBIAMOS LA CARPETA PARA EMPEZAR TOTALMENTE LIMPIOS
WORKSPACE_DIR = "../lightrag_graphs"
if not os.path.exists(WORKSPACE_DIR):
    os.mkdir(WORKSPACE_DIR)

# 1. El Cerebro (Generación de texto)
async def gemini_llm_wrapper(prompt, system_prompt=None, history_messages=[], **kwargs):
    kwargs["base_url"] = "https://generativelanguage.googleapis.com/v1beta/openai/"
    kwargs["api_key"] = os.getenv("GEMINI_API_KEY") 
    
    return await openai_complete_if_cache(
        model="gemini-2.5-flash-lite",
        prompt=prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        **kwargs
    )
@wrap_embedding_func_with_attrs(
    embedding_dim=3072, 
    max_token_size=2048,
    model_name="gemini-embedding-001" 
)
async def gemini_embedding_wrapper(texts: list[str]) -> np.ndarray:
    client = AsyncOpenAI(
        api_key=os.getenv("GEMINI_API_KEY"),
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )
    
    embeddings = []
    for text in texts:
        response = await client.embeddings.create(
            input=[text],  # siempre 1 texto
            model="gemini-embedding-001"
        )
        embeddings.append(response.data[0].embedding)
    
    return np.array(embeddings)  # shape: (len(texts), 768)

# 3. Ensamblamos el motor
rag = LightRAG(
    working_dir=WORKSPACE_DIR,
    llm_model_func=gemini_llm_wrapper,
    llm_model_max_async=1, 
    embedding_func=gemini_embedding_wrapper
)

print(f"✅ Motor ensamblado. Archivos guardados en: {WORKSPACE_DIR}")

INFO: [] Created new empty graph file: ../lightrag_graphs/graph_chunk_entity_relation.graphml
INFO:nano-vectordb:Init {'embedding_dim': 3072, 'metric': 'cosine', 'storage_file': '../lightrag_graphs/vdb_entities.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 3072, 'metric': 'cosine', 'storage_file': '../lightrag_graphs/vdb_relationships.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 3072, 'metric': 'cosine', 'storage_file': '../lightrag_graphs/vdb_chunks.json'} 0 data


⚙️ Configurando LightRAG con motor 100% Gemini...
✅ Motor ensamblado. Archivos guardados en: ../lightrag_graphs


In [28]:
with open(ARCHIVO_ACTUAL, "r", encoding="utf-8") as f:
    texto = f.read()
    
await rag.initialize_storages()
rag.insert(texto)

INFO: Created 1 duplicate document records with track_id: insert_20260312_172142_82d6560d
INFO: Preserving 4 failed document entries for manual review
INFO: Reset 2 documents from PROCESSING/FAILED to PENDING status
INFO: Processing 2 document(s)
INFO: Extracting stage 1/2: unknown_source
INFO: Processing d-id: doc-4a1a5b5ef8261b747e61e2dff849b1a6
INFO: Extracting stage 2/2: unknown_source
INFO: Processing d-id: doc-fac4bd790099efa96fe8be11a8edb416
INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)
INFO: LLM func: 1 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
INFO:  == LLM cache == saving: default:extract:b0bde80f005f7224529dce673c9671d9
INFO:  == LLM cache == saving: default:extract:ad5a0be74281dde6263eee2a56ab180a
INFO:  == LLM cache == saving: default:extract:6329220c244a6330ce6dc28892bf1b2a
INFO: Chunk 1 of 1 extracted 6 Ent + 7 Rel chunk-fac4bd790099efa96fe8be11a8edb416
INFO: Merging stage 2/2:

'insert_20260312_172142_82d6560d'

In [29]:
import os
import networkx as nx
from pyvis.network import Network
import webbrowser

ruta_grafo = os.path.join("../lightrag_graphs", "graph_chunk_entity_relation.graphml")

if not os.path.exists(ruta_grafo):
    print("No encuentro el archivo del grafo. Revisa que la ruta sea correcta.")
else:    
    G = nx.read_graphml(ruta_grafo)
    
    net = Network(notebook=False, width="100%", height="100vh", bgcolor="#222222", font_color="white")
    net.from_nx(G)
    net.repulsion(node_distance=150, central_gravity=0.2, spring_length=200)
    
    archivo_html = "grafo_scrooge_fullscreen.html"
    net.save_graph(archivo_html)
    
    ruta_absoluta = "file://" + os.path.abspath(archivo_html)
    webbrowser.open(ruta_absoluta)

In [30]:
import asyncio
from lightrag import QueryParam

print("🔍 Iniciando batería de pruebas sobre el Grafo de Dickens...")

async def hacer_preguntas():
    # 1. Búsqueda Local (Para detalles precisos y personajes secundarios)
    print("\n" + "="*50)
    print("--- 📍 BÚSQUEDA LOCAL (Detalles de personajes) ---")
    print("="*50)
    pregunta_1 = "¿Quién es Tiny Tim, qué problema físico tiene y cuál es su famosa frase final?"
    print(f"❓ Pregunta: {pregunta_1}\n")
    respuesta_local = await rag.aquery(pregunta_1, param=QueryParam(mode="local"))
    print(respuesta_local)

    # 2. Búsqueda Global (Para temas abstractos de todo el libro)
    print("\n" + "="*50)
    print("--- 🌍 BÚSQUEDA GLOBAL (Visión de conjunto) ---")
    print("="*50)
    pregunta_2 = "Resume brevemente la evolución psicológica de Scrooge a través de sus encuentros con los tres fantasmas."
    print(f"❓ Pregunta: {pregunta_2}\n")
    respuesta_global = await rag.aquery(pregunta_2, param=QueryParam(mode="global"))
    print(respuesta_global)

    # 3. Búsqueda Híbrida (La joya de la corona: Local + Global)
    print("\n" + "="*50)
    print("--- 🧠 BÚSQUEDA HÍBRIDA (Detalles + Contexto) ---")
    print("="*50)
    pregunta_3 = "¿Qué contraste existe entre el ambiente en el despacho de Scrooge al principio de la historia y el ambiente en la casa de los Cratchit durante la cena?"
    print(f"❓ Pregunta: {pregunta_3}\n")
    respuesta_hibrida = await rag.aquery(pregunta_3, param=QueryParam(mode="hybrid"))
    print(respuesta_hibrida)

# Ejecutamos las preguntas
await hacer_preguntas()

🔍 Iniciando batería de pruebas sobre el Grafo de Dickens...

--- 📍 BÚSQUEDA LOCAL (Detalles de personajes) ---
❓ Pregunta: ¿Quién es Tiny Tim, qué problema físico tiene y cuál es su famosa frase final?



INFO:  == LLM cache == saving: local:keywords:baef0e875feeabfadc50e2851df12d78
INFO: Query nodes: Tiny Tim, Disability, Catchphrase (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 115 relations
INFO: Raw search results: 40 entities, 115 relations, 0 vector chunks
INFO: After truncation: 40 entities, 115 relations
INFO: Selecting 16 from 16 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 115 relations
INFO: Round-robin merged chunks: 16 -> 16 (deduplicated 0)
INFO: Final context: 40 entities, 115 relations, 16 chunks
INFO: Final chunks S+F/O: E9/1 E9/2 E21/3 E3/4 E2/5 E2/6 E3/7 E1/8 E1/9 E1/10 E2/11 E2/12 E2/13 E1/14 E1/15 E1/16
INFO:  == LLM cache == saving: local:query:7a1aff074d8d4122dac8b4e8ffa18b49


Tiny Tim es el hijo de Bob y Mrs. Cratchit. Es descrito como un niño con problemas físicos, ya que utiliza una muleta y sus extremidades están sostenidas por un armazón de hierro. A pesar de sus limitaciones, se le representa como un niño que está creciendo fuerte y sano.

Su famosa frase final es: "¡Dios nos bendiga a todos!".

También se menciona que Tiny Tim esperaba que la gente lo recordara en la iglesia porque era un "inválido" y que recordaran quién hacía "andar a los mendigos cojos" y "ver a los ciegos".

### References

* [0] document
* [1] document
* [3] document
* [4] document

--- 🌍 BÚSQUEDA GLOBAL (Visión de conjunto) ---
❓ Pregunta: Resume brevemente la evolución psicológica de Scrooge a través de sus encuentros con los tres fantasmas.



INFO:  == LLM cache == saving: global:keywords:0cbe5ddea94f175f95209bcf37e53e61
INFO: Query edges: Evolución psicológica, Personajes (top_k:40, cosine:0.2)
INFO: Global query: 54 entites, 40 relations
INFO: Raw search results: 54 entities, 40 relations, 0 vector chunks
INFO: After truncation: 54 entities, 40 relations
INFO: Selecting 38 from 38 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 40 relations
INFO: Round-robin merged chunks: 38 -> 38 (deduplicated 0)
INFO: Final context: 54 entities, 40 relations, 17 chunks
INFO: Final chunks S+F/O: E1/1 E1/2 E4/3 E1/4 E5/5 E7/6 E3/7 E12/8 E3/9 E1/10 E1/11 E1/12 E1/13 E2/14 E4/15 E4/16 E4/17
INFO:  == LLM cache == saving: global:query:f4d1cbcd2dd9ebae676099db2983d504


A lo largo de sus encuentros con los tres fantasmas, Ebenezer Scrooge experimenta una profunda transformación psicológica.

Inicialmente, Scrooge es retratado como un hombre de negocios avaro, frío y solitario, que desprecia la Navidad y carece de imaginación. Su vida está marcada por la rutina y la falta de empatía hacia los demás, centrándose únicamente en la acumulación de riqueza.

El primer fantasma, el del Pasado, le muestra visiones de su infancia y juventud, revelando las causas de su aislamiento y la pérdida de su amor a causa de su creciente codicia. Estas visiones evocan en él sentimientos de nostalgia y arrepentimiento.

El segundo fantasma, el del Presente, le exhibe la alegría y la generosidad de la Navidad, contrastando su propia soledad con la calidez de las celebraciones ajenas, como la de su sobrino y la humilde familia de Bob Cratchit. A través de estas escenas, Scrooge empieza a vislumbrar la importancia de la conexión humana y la compasión.

Finalmente, el tercer f

INFO:  == LLM cache == saving: hybrid:keywords:b17323a8575c9ebaa249ad40c3c908e3
INFO: Query nodes: Ebenezer Scrooge, Scrooge's office, Cratchit family, Christmas dinner, Victorian England (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 490 relations
INFO: Query edges: Literary comparison, Character environment, Social commentary (top_k:40, cosine:0.2)
INFO: Global query: 60 entites, 40 relations
INFO: Raw search results: 97 entities, 518 relations, 0 vector chunks
INFO: After truncation: 97 entities, 129 relations
INFO: Selecting 38 from 38 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 129 relations
INFO: Round-robin merged chunks: 38 -> 38 (deduplicated 0)
INFO: Final context: 97 entities, 129 relations, 12 chunks
INFO: Final chunks S+F/O: E4/1 E12/2 E8/3 E5/4 E4/5 E2/6 E10/7 E10/8 E7/9 E5/10 E5/11 E10/12
INFO:  == LLM cache == saving: hybrid:query:37373f05299148fffe5fcabf79841ac1


Al principio de la historia, el despacho de Scrooge se describe como un lugar sombrío y frío. Scrooge trabaja allí en un ambiente de niebla y oscuridad, con un fuego muy pequeño que apenas proporciona calor. Su empleado, Bob Cratchit, trabaja en una celda pequeña y miserable, con un fuego aún más pequeño, ya que Scrooge guarda la caja de carbón en su propia habitación y se niega a que el empleado la reponga. Scrooge llega temprano a su despacho para observar a Bob Cratchit, quien llega tarde y se disculpa diciendo que estuvo celebrando la víspera de Navidad. Scrooge, a pesar de que es solo una vez al año, se queja de pagar un día de salario por trabajo no realizado y advierte al empleado que si vuelve a oír algo más, perderá su puesto.

En contraste, la casa de los Cratchit durante la cena, a la que el Espíritu de la Navidad Presente bendice, es un lugar cálido y afectuoso, a pesar de su modesta vivienda de cuatro habitaciones. La Sra. Cratchit, su esposa, viste bien con cintas, y sus 